In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



# ***IMPORTS***

In [6]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mnv3_prep
from tensorflow.keras.applications.convnext import preprocess_input as convnext_prep
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as effv2_prep
from tensorflow.keras.applications.densenet import preprocess_input as densenet_prep


# ***CONFIG***

In [7]:
# Each model has its own IMG_SIZE — must match training exactly
IMG_SIZE_384 = 384    # MobileNetV3, ConvNeXt-Tiny, EfficientNetV2-S
IMG_SIZE_224 = 224    # DenseNet201

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

# ***IMAGE PATHS***

In [8]:
IMAGE_PATHS = {
    "glioma"     : "/content/drive/MyDrive/Project work/Test Data/test_glioma.jpg",
    "meningioma" : "/content/drive/MyDrive/Project work/Test Data/test_meningioma.jpg",
    "pituitary"  : "/content/drive/MyDrive/Project work/Test Data/test_pituitary.jpg",
    "notumor"    : "/content/drive/MyDrive/Project work/Test Data/test_no tumor.jpg"
}

# ***MODEL PATHS***

In [14]:
MODEL_PATHS = {
    "MobileNetV3"     : "/content/drive/MyDrive/Project work/models/class_Tumor_mobilenet_v3.keras",
    "ConvNeXt-Tiny"   : "/content/drive/MyDrive/Project work/models/class_Tumor_convnext_tiny_tumor.keras",
    "EfficientNetV2-S": "/content/drive/MyDrive/Project work/models/class_Tumor_v2s_clean.keras",
    "DenseNet201"     : "/content/drive/MyDrive/Project work/models/class_Tumor_densenet_201.keras",
}
import os

for name, path in MODEL_PATHS.items():
    print(f"{name}: {os.path.exists(path)}")

MobileNetV3: True
ConvNeXt-Tiny: True
EfficientNetV2-S: True
DenseNet201: True


# ***IMAGE LOADER***

In [15]:
def load_image(path, img_size):
    img = tf.keras.utils.load_img(path, target_size=(img_size, img_size))
    img = tf.keras.utils.img_to_array(img)
    img = tf.expand_dims(img, axis=0)
    return img

# ***PREPROCESS MAP***

In [16]:

PREPROCESS = {
    "MobileNetV3"     : mnv3_prep,
    "ConvNeXt-Tiny"   : convnext_prep,
    "EfficientNetV2-S": effv2_prep,
    "DenseNet201"     : densenet_prep,
}

IMG_SIZES = {
    "MobileNetV3"     : IMG_SIZE_384,
    "ConvNeXt-Tiny"   : IMG_SIZE_384,
    "EfficientNetV2-S": IMG_SIZE_384,
    "DenseNet201"     : IMG_SIZE_224,
}


# ***LOAD MODELS***

In [17]:
models = {}

for name, path in MODEL_PATHS.items():
    print(f"Loading {name}...")
    models[name] = tf.keras.models.load_model(path)

print("All models loaded successfully.\n")


Loading MobileNetV3...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 93 variables whereas the saved optimizer has 184 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loading ConvNeXt-Tiny...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 96 variables whereas the saved optimizer has 190 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loading EfficientNetV2-S...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 232 variables whereas the saved optimizer has 462 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loading DenseNet201...
All models loaded successfully.



/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 311 variables whereas the saved optimizer has 620 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


# ***TESTING***

In [18]:

print("🧠 Tumor Classification – Manual Image Testing")
print("=" * 60)

for true_label, img_path in IMAGE_PATHS.items():

    print(f"\nTesting image: {true_label}")
    print("-" * 60)

    for model_name, model in models.items():

        img      = load_image(img_path, IMG_SIZES[model_name])
        img_proc = PREPROCESS[model_name](img)

        preds      = model.predict(img_proc, verbose=0)[0]
        idx        = np.argmax(preds)
        pred_label = CLASS_NAMES[idx]
        confidence = preds[idx]

        correct = "✅" if pred_label == true_label else "❌"

        print(
            f"  {correct} {model_name:18s} → "
            f"Pred: {pred_label:12s} | "
            f"Conf: {confidence:.4f}"
        )

🧠 Tumor Classification – Manual Image Testing

Testing image: glioma
------------------------------------------------------------
  ✅ MobileNetV3        → Pred: glioma       | Conf: 0.7437
  ✅ ConvNeXt-Tiny      → Pred: glioma       | Conf: 0.4528
  ✅ EfficientNetV2-S   → Pred: glioma       | Conf: 0.9657
  ❌ DenseNet201        → Pred: pituitary    | Conf: 0.3233

Testing image: meningioma
------------------------------------------------------------
  ✅ MobileNetV3        → Pred: meningioma   | Conf: 0.8019
  ✅ ConvNeXt-Tiny      → Pred: meningioma   | Conf: 0.5716
  ✅ EfficientNetV2-S   → Pred: meningioma   | Conf: 0.8944
  ❌ DenseNet201        → Pred: pituitary    | Conf: 0.3725

Testing image: pituitary
------------------------------------------------------------
  ❌ MobileNetV3        → Pred: meningioma   | Conf: 0.5737
  ✅ ConvNeXt-Tiny      → Pred: pituitary    | Conf: 0.4762
  ✅ EfficientNetV2-S   → Pred: pituitary    | Conf: 0.8575
  ✅ DenseNet201        → Pred: pituitary    | 